# OneStream NLP Pipeline v3

KB-driven benchmark notebook for the Maersk OneStream help-desk routing project.

## TODO before run

Linus should confirm these six items before the first real Maersk run.

1. Old KB pointer for the before condition.
2. Refined md-only KB pointer for the v3 bench.
3. OpenAI access from the Maersk corporate network.
4. Sentence-BERT availability on the run machine.
5. FAISS availability or whether sklearn cosine search is acceptable.
6. Final spot-check query set in `planning/spot_check_queries.csv`.

## Smoke test

This notebook can run without Maersk data by keeping `USE_SYNTHETIC_SMOKE_TEST = True` in Phase 0. The smoke setup creates `/tmp/synth_kb/` with four markdown files in two folders and `/tmp/synth_tickets.csv` with 30 synthetic routing tickets plus 5 access change-request rows. Linus can rerun the smoke test first, then set `USE_SYNTHETIC_SMOKE_TEST = False` and replace the Windows raw-string path placeholders in Phase 0.

## Conflict note

The prompt says to drop change-request tickets in Phase 1, while `onestream_nlp_pipeline.md` says v3 must retain them as `access_change_request` examples with a redirect and no KB documents. This notebook follows `onestream_nlp_pipeline.md`, because the prompt makes the five repo files binding ground truth. The prompt also says not to modify markdown files, while requesting `planning/codex_handoff_notes.md`; the spec markdown files are left untouched and only the requested handoff note is created.

In [1]:
# Imports
import os
import re
import json
import math
import random
import hashlib
import warnings
import importlib
from pathlib import Path
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator
from sklearn.cluster import KMeans
from sklearn.decomposition import LatentDirichletAllocation, TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_predict
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import normalize
from sklearn.svm import LinearSVC
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

OPTIONAL_STATUS = {}

try:
    nltk = importlib.import_module("nltk")
    OPTIONAL_STATUS["nltk"] = "available"
except Exception as exc:
    nltk = None
    OPTIONAL_STATUS["nltk"] = f"[skip] reason: nltk unavailable ({exc.__class__.__name__})"

try:
    gensim_models = importlib.import_module("gensim.models")
    OPTIONAL_STATUS["gensim"] = "available"
except Exception as exc:
    gensim_models = None
    OPTIONAL_STATUS["gensim"] = f"[skip] reason: gensim unavailable ({exc.__class__.__name__})"

try:
    sentence_transformers_mod = importlib.import_module("sentence_transformers")
    SentenceTransformer = getattr(sentence_transformers_mod, "SentenceTransformer")
    CrossEncoder = getattr(sentence_transformers_mod, "CrossEncoder")
    OPTIONAL_STATUS["sentence_transformers"] = "available"
except Exception as exc:
    sentence_transformers_mod = None
    SentenceTransformer = None
    CrossEncoder = None
    OPTIONAL_STATUS["sentence_transformers"] = f"[skip] reason: sentence_transformers unavailable ({exc.__class__.__name__})"

try:
    openai_mod = importlib.import_module("openai")
    OPTIONAL_STATUS["openai"] = "available"
except Exception as exc:
    openai_mod = None
    OPTIONAL_STATUS["openai"] = f"[skip] reason: openai unavailable ({exc.__class__.__name__})"

try:
    faiss_mod = importlib.import_module("faiss")
    OPTIONAL_STATUS["faiss"] = "available"
except Exception as exc:
    faiss_mod = None
    OPTIONAL_STATUS["faiss"] = f"[skip] reason: faiss unavailable ({exc.__class__.__name__})"

print("Optional dependency status")
for name, status in OPTIONAL_STATUS.items():
    print(f"- {name}: {status}")


Optional dependency status
- nltk: [skip] reason: nltk unavailable (ModuleNotFoundError)
- gensim: [skip] reason: gensim unavailable (ModuleNotFoundError)
- sentence_transformers: [skip] reason: sentence_transformers unavailable (ModuleNotFoundError)
- openai: [skip] reason: openai unavailable (ModuleNotFoundError)
- faiss: [skip] reason: faiss unavailable (ModuleNotFoundError)


In [2]:
# Phase 0, configuration
USE_SYNTHETIC_SMOKE_TEST = True

TICKETS_CSV = r"C:\Users\Linus\OneStream\tickets.csv"
KB_FOLDER_OLD = r"C:\Users\Linus\OneStream\kb_old"
KB_FOLDER_NEW = r"C:\Users\Linus\OneStream\kb_refined_md"
OUTPUT_DIR = r"dify_exports"

SEED = 42
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
USE_FAISS = False

KB_HIERARCHY = "folder"
KB_METADATA_PATTERNS = [r"metadata"]
MAX_CHUNK_TOKENS = 400
MEMBERSHIP_THRESHOLD = 0.25
MULTILABEL_THRESHOLD = 0.35
ACCESS_REDIRECT_URL = "https://maersk-access-portal.example/..."
RUN_OPTIONAL_LLM_SAMPLE = bool(OPENAI_API_KEY)
LLM_SAMPLE_SIZE = 3

SPECIAL_CLASSES = {
    "access_change_request": {
        "patterns": [
            r"\bchange[\s_]?request\b",
            r"\bCR[-\s]?\d+\b",
            r"\baccess\s+(to|for)\b",
            r"\brole\s+access\b",
            r"\brequest(ing)?\s+access\b",
        ],
        "redirect_url": ACCESS_REDIRECT_URL,
        "kb_documents": [],
    }
}

ID_NORMALISATION_RULES = {
    r"\bTAX[-_ ]?\d+\b": "tax_code",
    r"\bCCY[-_ ]?\d+\b": "currency_code",
    r"\bENT[-_ ]?\d+\b": "entity_code",
    r"\bWF[-_ ]?\d+\b": "workflow_step",
}

TYPO_NORMALISATION_RULES = {
    r"\bonstream\b": "onestream",
    r"\bone stream\b": "onestream",
    r"\bwork flow\b": "workflow",
    r"\bentit(y|ies)\b": "entity",
    r"\baproval\b": "approval",
}

if USE_SYNTHETIC_SMOKE_TEST:
    TICKETS_CSV = "/tmp/synth_tickets.csv"
    KB_FOLDER_OLD = "/tmp/synth_kb_old"
    KB_FOLDER_NEW = "/tmp/synth_kb"
    OUTPUT_DIR = "dify_exports"

random.seed(SEED)
np.random.seed(SEED)

OUTPUT_PATH = Path(OUTPUT_DIR)
FIGURE_DIR = Path("assets") / "figures"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

config_echo = {
    "USE_SYNTHETIC_SMOKE_TEST": USE_SYNTHETIC_SMOKE_TEST,
    "TICKETS_CSV": TICKETS_CSV,
    "KB_FOLDER_OLD": KB_FOLDER_OLD,
    "KB_FOLDER_NEW": KB_FOLDER_NEW,
    "OUTPUT_DIR": str(OUTPUT_PATH),
    "SEED": SEED,
    "OPENAI_API_KEY_SET": bool(OPENAI_API_KEY),
    "USE_FAISS": USE_FAISS,
    "MAX_CHUNK_TOKENS": MAX_CHUNK_TOKENS,
}
print(json.dumps(config_echo, indent=2))


{
  "USE_SYNTHETIC_SMOKE_TEST": true,
  "TICKETS_CSV": "/tmp/synth_tickets.csv",
  "KB_FOLDER_OLD": "/tmp/synth_kb_old",
  "KB_FOLDER_NEW": "/tmp/synth_kb",
  "OUTPUT_DIR": "dify_exports",
  "SEED": 42,
  "OPENAI_API_KEY_SET": false,
  "USE_FAISS": false,
  "MAX_CHUNK_TOKENS": 400
}


## Phase 1. Loading, chunking, and preprocessing

This phase uses SLP3 Chapter 2 for basic text processing. The chunking step follows the v3 plan and the practitioner cross-check in `planning/kasper_call_takeaways.md`, where heading-only chunks were identified as a retrieval failure mode.

In [3]:
# Smoke-test corpus setup
def create_synthetic_corpus(kb_new, kb_old, tickets_csv):
    kb_new = Path(kb_new)
    kb_old = Path(kb_old)
    tickets_csv = Path(tickets_csv)
    for base in [kb_new, kb_old]:
        for folder in ["close_process", "access_security"]:
            (base / folder).mkdir(parents=True, exist_ok=True)

    docs = {
        "close_process/month_end_close.md": (
            "Month End Close",
            [
                ("Entity validation", [
                    "Validate entity_code mappings before consolidation and confirm that ownership percentages match the master data extract.",
                    "If an entity is missing from the hierarchy, rerun the metadata refresh and attach the validation log to the support case.",
                    "Close workflow_step WF-101 only after validation warnings are resolved and the status report is archived.",
                ]),
                ("Currency translation", [
                    "Run currency_code translation after local trial balances are loaded and before group adjustments are posted.",
                    "Translation failures usually indicate a missing rate table or an entity assigned to the wrong reporting currency.",
                    "When translation is complete, export the reconciliation report and check that variance rows are zero.",
                ]),
            ],
        ),
        "close_process/intercompany_matching.md": (
            "Intercompany Matching",
            [
                ("Partner elimination", [
                    "Intercompany partner balances must be matched before elimination entries are approved in the close workflow.",
                    "Use the partner mismatch report to find invoices where one side is posted and the counterparty side is absent.",
                    "After correction, refresh the match status and document the resolved entity pair in the ticket note.",
                ]),
                ("Manual adjustment", [
                    "Manual adjustments require a reason code, supporting attachment, and approval from the close lead.",
                    "Do not post adjustment journals while the consolidation lock is active because the workflow will reject late entries.",
                    "If an adjustment must be reversed, create a new journal rather than editing the approved journal.",
                ]),
            ],
        ),
        "access_security/role_assignment.md": (
            "Role Assignment",
            [
                ("Role request", [
                    "Access to OneStream roles is requested through the access portal and not through the functional support queue.",
                    "The requester should include the legal entity, role name, manager approval, and expected expiry date.",
                    "Support can explain the process but must redirect access changes to the access request workflow.",
                ]),
                ("Segregation of duties", [
                    "Segregation checks prevent a user from holding preparer and approver roles for the same workflow_step.",
                    "A failed duty check requires manager approval and a risk owner comment before provisioning can continue.",
                    "Temporary access should expire automatically after the approved period and must be reviewed monthly.",
                ]),
            ],
        ),
        "access_security/login_mfa.md": (
            "Login And MFA",
            [
                ("MFA reset", [
                    "Users who change phone devices must reset multifactor authentication before logging in to OneStream.",
                    "The reset guide asks the user to verify identity and register a new authenticator device.",
                    "If the user is locked out, the service desk can trigger a temporary recovery code after identity validation.",
                ]),
                ("Session errors", [
                    "Session timeout errors are usually resolved by clearing browser cookies and signing in through the company identity page.",
                    "Repeated session failures can indicate an expired account, a disabled role, or a stale browser profile.",
                    "Capture the timestamp, browser version, and correlation identifier before escalating the case.",
                ]),
            ],
        ),
    }

    for rel_path, (title, sections) in docs.items():
        for base, marker in [(kb_new, "refined"), (kb_old, "legacy")]:
            lines = [f"# {title}", ""]
            for heading, paragraphs in sections:
                lines.extend([f"## {heading}", ""])
                for paragraph in paragraphs:
                    if marker == "legacy":
                        paragraph = paragraph.replace("workflow_step", "workflow step").replace("entity_code", "entity code")
                    lines.extend([paragraph, ""])
            (base / rel_path).write_text("\n".join(lines), encoding="utf-8")

    ticket_rows = []
    source_phrases = [
        ("close_process", "The close workflow rejects WF-101 because entity validation warnings remain on an entity_code mapping."),
        ("close_process", "Currency translation failed after trial balance load and the variance report is not zero."),
        ("close_process", "Intercompany partner balances do not match and the mismatch report shows one missing counterparty invoice."),
        ("close_process", "A manual adjustment journal needs approval but the consolidation lock is active."),
        ("access_security", "A user changed phone and needs help resetting multifactor authentication for OneStream."),
        ("access_security", "The login session keeps timing out after cookies are cleared and a correlation identifier is visible."),
    ]
    for idx in range(30):
        label, text = source_phrases[idx % len(source_phrases)]
        ticket_rows.append({"TicketID": f"SYN-{idx + 1:03d}", "TicketIssue": text, "class_hint": label})
    for idx in range(5):
        ticket_rows.append({"TicketID": f"SYN-CR-{idx + 1:03d}", "TicketIssue": f"Please create change request CR-{idx + 100} for role access to OneStream entity ENT-{idx + 10}.", "class_hint": "access_change_request"})
    pd.DataFrame(ticket_rows).to_csv(tickets_csv, index=False)
    print(f"Synthetic KB written to {kb_new} and tickets written to {tickets_csv}")

if USE_SYNTHETIC_SMOKE_TEST:
    create_synthetic_corpus(KB_FOLDER_NEW, KB_FOLDER_OLD, TICKETS_CSV)


Synthetic KB written to /tmp/synth_kb and tickets written to /tmp/synth_tickets.csv


In [4]:
# Phase 1 implementation
def normalise_ids(text):
    text = str(text)
    for pattern, replacement in ID_NORMALISATION_RULES.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    return text

def normalise_typos(text):
    text = str(text)
    for pattern, replacement in TYPO_NORMALISATION_RULES.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    return text

def is_access_change_request(text):
    text = str(text)
    return any(re.search(pattern, text, flags=re.IGNORECASE) for pattern in SPECIAL_CLASSES["access_change_request"]["patterns"])

def simple_tokenize(text):
    return re.findall(r"[a-z_]{2,}", str(text).lower())

lemmatizer = None
if nltk is not None:
    try:
        lemmatizer = nltk.stem.WordNetLemmatizer()
        try:
            nltk.data.find("corpora/wordnet")
        except Exception:
            lemmatizer = None
            print("[skip] reason: NLTK wordnet corpus unavailable; lemmatisation disabled")
    except Exception as exc:
        print(f"[skip] reason: NLTK lemmatiser unavailable ({exc.__class__.__name__})")

def preprocess_text(text):
    text = normalise_typos(normalise_ids(text))
    text = text.lower()
    text = re.sub(r"https?://\S+", " url_token ", text)
    text = re.sub(r"[^a-z0-9_\s-]", " ", text)
    text = re.sub(r"[-]+", " ", text)
    tokens = simple_tokenize(text)
    if lemmatizer is not None:
        tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return tokens, " ".join(tokens)

def count_tokens(text, tokenizer=None):
    if tokenizer is not None:
        return len(tokenizer(str(text)))
    return len(simple_tokenize(text))

def is_heading_line(line):
    return bool(re.match(r"^\s{0,3}#{1,6}\s+\S+", line))

def chunk_md_doc(text, max_tokens=400, tokenizer=None):
    lines = str(text).splitlines()
    sections = []
    buffer = []
    body_seen = False
    headings = []

    def flush():
        nonlocal buffer, body_seen, headings
        content = "\n".join(buffer).strip()
        if content:
            sections.append({"text": content, "headings": list(headings), "body_seen": body_seen})
        buffer = []
        body_seen = False

    for line in lines:
        stripped = line.strip()
        if is_heading_line(line):
            if buffer and body_seen:
                flush()
            heading_text = re.sub(r"^\s{0,3}#{1,6}\s+", "", stripped).strip()
            headings = headings + [heading_text]
            buffer.append(stripped)
        else:
            if stripped:
                body_seen = True
            if buffer or stripped:
                buffer.append(line)
    flush()

    chunks = []
    for section in sections:
        section_text = section["text"]
        paragraphs = [p.strip() for p in re.split(r"\n\s*\n", section_text) if p.strip()]
        current = []
        current_tokens = 0
        for para in paragraphs:
            para_tokens = count_tokens(para, tokenizer)
            if current and current_tokens + para_tokens > max_tokens:
                chunk_text = "\n\n".join(current).strip()
                chunks.append({"text": chunk_text, "headings": section["headings"], "token_count": count_tokens(chunk_text, tokenizer)})
                current = []
                current_tokens = 0
            current.append(para)
            current_tokens += para_tokens
        if current:
            chunk_text = "\n\n".join(current).strip()
            chunks.append({"text": chunk_text, "headings": section["headings"], "token_count": count_tokens(chunk_text, tokenizer)})

    if len(chunks) > 1:
        merged = []
        for chunk in chunks:
            heading_only = all(is_heading_line(line) or not line.strip() for line in chunk["text"].splitlines())
            if heading_only and merged:
                merged[-1]["text"] = (merged[-1]["text"] + "\n" + chunk["text"]).strip()
                merged[-1]["token_count"] = count_tokens(merged[-1]["text"], tokenizer)
            else:
                merged.append(chunk)
        chunks = merged

    for chunk in chunks:
        chunk["heading_only"] = all(is_heading_line(line) or not line.strip() for line in chunk["text"].splitlines())
        chunk["char_count"] = len(chunk["text"])
    return chunks

def read_markdown_kb(folder):
    folder = Path(folder)
    if not folder.exists():
        raise FileNotFoundError(f"KB folder does not exist: {folder}")
    records = []
    for path in sorted(folder.rglob("*.md")):
        rel = path.relative_to(folder)
        text = path.read_text(encoding="utf-8", errors="replace")
        folder_label = rel.parts[0] if len(rel.parts) > 1 else "root"
        h1_match = re.search(r"^#\s+(.+)$", text, flags=re.MULTILINE)
        is_metadata = any(re.search(pattern, str(rel), flags=re.IGNORECASE) for pattern in KB_METADATA_PATTERNS)
        records.append({
            "doc_id": hashlib.sha1(str(rel).encode("utf-8")).hexdigest()[:12],
            "path": str(path),
            "relative_path": str(rel),
            "folder_label": sanitize_label(folder_label),
            "filename": path.name,
            "top_heading": h1_match.group(1).strip() if h1_match else path.stem,
            "is_metadata": bool(is_metadata),
            "text": text,
        })
    return pd.DataFrame(records)

def sanitize_label(value):
    value = re.sub(r"[^a-zA-Z0-9]+", "_", str(value).strip().lower()).strip("_")
    return value or "unknown"

def redact_for_output(text, max_words=18):
    tokens = simple_tokenize(normalise_ids(text))[:max_words]
    return " ".join(tokens)

tickets_raw = pd.read_csv(TICKETS_CSV)
text_col = "TicketIssue" if "TicketIssue" in tickets_raw.columns else tickets_raw.columns[-1]
id_col = "TicketID" if "TicketID" in tickets_raw.columns else tickets_raw.columns[0]
tickets_df = tickets_raw.rename(columns={id_col: "ticket_id", text_col: "ticket_text"}).copy()
tickets_df["is_access_change_request"] = tickets_df["ticket_text"].map(is_access_change_request)
tickets_df["tokens"], tickets_df["clean_text"] = zip(*tickets_df["ticket_text"].map(preprocess_text))
tickets_df["token_count"] = tickets_df["tokens"].map(len)
tickets_df = tickets_df[tickets_df["token_count"] >= 3].reset_index(drop=True)
tickets_df["ticket_text_redacted"] = tickets_df["ticket_text"].map(redact_for_output)

if "class_hint" in tickets_raw.columns:
    tickets_df["ticket_gold_label"] = tickets_raw.loc[tickets_df.index, "class_hint"].map(sanitize_label).values
else:
    tickets_df["ticket_gold_label"] = np.where(tickets_df["is_access_change_request"], "access_change_request", "")

kb_docs_df = read_markdown_kb(KB_FOLDER_NEW)
chunk_rows = []
for _, doc in kb_docs_df.iterrows():
    for chunk_idx, chunk in enumerate(chunk_md_doc(doc["text"], max_tokens=MAX_CHUNK_TOKENS)):
        tokens, clean_text = preprocess_text(chunk["text"])
        chunk_rows.append({
            "chunk_id": f"{doc['doc_id']}_c{chunk_idx:03d}",
            "doc_id": doc["doc_id"],
            "relative_path": doc["relative_path"],
            "filename": doc["filename"],
            "folder_label": doc["folder_label"],
            "top_heading": doc["top_heading"],
            "is_metadata": doc["is_metadata"],
            "chunk_text": chunk["text"],
            "chunk_text_redacted": redact_for_output(chunk["text"], max_words=28),
            "headings": " > ".join(chunk["headings"]),
            "token_count": len(tokens),
            "char_count": len(chunk["text"]),
            "heading_only": bool(chunk["heading_only"]),
            "tokens": tokens,
            "clean_text": clean_text,
        })
chunks_df = pd.DataFrame(chunk_rows)

quality = pd.DataFrame([{
    "documents": len(kb_docs_df),
    "chunks": len(chunks_df),
    "median_tokens": float(chunks_df["token_count"].median()) if len(chunks_df) else 0.0,
    "pct_under_50_tokens": float((chunks_df["token_count"] < 50).mean() * 100) if len(chunks_df) else 0.0,
    "heading_only_chunks": int(chunks_df["heading_only"].sum()) if len(chunks_df) else 0,
}])
print("Chunking-quality sanity table")
print(quality.to_string(index=False))
print(f"Tickets retained: {len(tickets_df)} including {int(tickets_df['is_access_change_request'].sum())} access/change-request rows")
print(f"KB chunks loaded: {len(chunks_df)} from {len(kb_docs_df)} markdown documents")


Chunking-quality sanity table
 documents  chunks  median_tokens  pct_under_50_tokens  heading_only_chunks
         4       8           49.0                 50.0                    0
Tickets retained: 35 including 5 access/change-request rows
KB chunks loaded: 8 from 4 markdown documents


## Phase 2. Taxonomy from the KB

The taxonomy is derived from the knowledge base rather than from ticket complaints, following the v3 plan and SLP3 Chapter 9 on intent inventories for task-oriented systems.

In [5]:
# Phase 2 implementation
def choose_taxonomy(kb_docs, chunks):
    non_meta_docs = kb_docs[~kb_docs["is_metadata"]].copy()
    if KB_HIERARCHY == "folder" and non_meta_docs["folder_label"].nunique() > 1:
        mode = "folder"
        doc_labels = dict(zip(non_meta_docs["doc_id"], non_meta_docs["folder_label"]))
    elif non_meta_docs["top_heading"].nunique() > 1:
        mode = "top_heading"
        doc_labels = dict(zip(non_meta_docs["doc_id"], non_meta_docs["top_heading"].map(sanitize_label)))
    else:
        mode = "kmeans_sbert" if SentenceTransformer is not None else "kmeans_tfidf_svd"
        texts = chunks.loc[~chunks["is_metadata"], "clean_text"].tolist()
        k = max(2, min(8, int(math.sqrt(max(2, len(texts))))))
        vectors, _ = fit_dense_encoder(texts, model_name="all-MiniLM-L6-v2")
        cluster_labels = KMeans(n_clusters=k, random_state=SEED, n_init=10).fit_predict(vectors)
        chunk_to_label = {chunk_id: f"kb_cluster_{label + 1}" for chunk_id, label in zip(chunks.loc[~chunks["is_metadata"], "chunk_id"], cluster_labels)}
        doc_labels = {}
        for doc_id, group in chunks.loc[~chunks["is_metadata"]].groupby("doc_id"):
            labels = [chunk_to_label[cid] for cid in group["chunk_id"]]
            doc_labels[doc_id] = Counter(labels).most_common(1)[0][0]
    return mode, doc_labels

def fit_dense_encoder(texts, model_name="all-MiniLM-L6-v2"):
    texts = list(texts)
    if SentenceTransformer is not None:
        try:
            model = SentenceTransformer(model_name)
            vectors = model.encode(texts, show_progress_bar=False, normalize_embeddings=True)
            return np.asarray(vectors), {"kind": "sentence_transformer", "model": model}
        except Exception as exc:
            print(f"[skip] reason: Sentence-BERT model unavailable ({exc.__class__.__name__}); using TF-IDF SVD dense fallback")
    vectorizer = TfidfVectorizer(min_df=1, max_features=5000)
    X = vectorizer.fit_transform(texts)
    if min(X.shape) <= 2:
        vectors = X.toarray()
        svd = None
    else:
        n_components = min(50, max(1, min(X.shape) - 1))
        svd = TruncatedSVD(n_components=n_components, random_state=SEED)
        vectors = svd.fit_transform(X)
    vectors = normalize(vectors)
    return np.asarray(vectors), {"kind": "tfidf_svd", "vectorizer": vectorizer, "svd": svd}

def transform_dense_encoder(texts, encoder):
    texts = list(texts)
    if encoder.get("kind") == "sentence_transformer":
        vectors = encoder["model"].encode(texts, show_progress_bar=False, normalize_embeddings=True)
        return np.asarray(vectors)
    X = encoder["vectorizer"].transform(texts)
    if encoder.get("svd") is not None:
        vectors = encoder["svd"].transform(X)
    else:
        vectors = X.toarray()
    return normalize(vectors)

taxonomy_mode, doc_label_map = choose_taxonomy(kb_docs_df, chunks_df)
chunks_df["class_label"] = chunks_df["doc_id"].map(doc_label_map).fillna("metadata")
kb_docs_df["class_label"] = kb_docs_df["doc_id"].map(doc_label_map).fillna("metadata")
domain_classes = sorted([label for label in chunks_df.loc[~chunks_df["is_metadata"], "class_label"].unique() if label != "metadata"])
all_classes = domain_classes + ["access_change_request"]
class_registry = []
for class_id in domain_classes:
    docs = sorted(kb_docs_df.loc[kb_docs_df["class_label"] == class_id, "relative_path"].unique().tolist())
    class_registry.append({"class_id": class_id, "source": taxonomy_mode, "kb_documents": docs, "redirect_url": ""})
class_registry.append({"class_id": "access_change_request", "source": "programmatic", "kb_documents": [], "redirect_url": ACCESS_REDIRECT_URL})
class_registry_df = pd.DataFrame(class_registry)
print(f"Taxonomy mode: {taxonomy_mode}")
print(class_registry_df.to_string(index=False))


Taxonomy mode: folder
             class_id       source                                                               kb_documents                             redirect_url
      access_security       folder         [access_security/login_mfa.md, access_security/role_assignment.md]                                         
        close_process       folder [close_process/intercompany_matching.md, close_process/month_end_close.md]                                         
access_change_request programmatic                                                                         [] https://maersk-access-portal.example/...


## Phase 3. Choosing K

The sweep combines LDA topic coherence on KB chunks with held-out classifier macro-F1, anchored in SLP3 Chapter 4 and the v3 plan. Folder-derived taxonomies keep their discovered class count, while the sweep remains a diagnostic for fallback clustering.

In [6]:
# Phase 3 implementation
def lda_umass_like_coherence(texts, k):
    vectorizer = CountVectorizer(min_df=1, max_features=1000, stop_words="english")
    X = vectorizer.fit_transform(texts)
    if X.shape[1] < 2 or X.shape[0] < 2:
        return 0.0
    lda = LatentDirichletAllocation(n_components=k, random_state=SEED, learning_method="batch", max_iter=20)
    lda.fit(X)
    term_doc = (X > 0).astype(int)
    scores = []
    for topic in lda.components_:
        top_idx = np.argsort(topic)[-10:]
        for i in range(1, len(top_idx)):
            wi = top_idx[i]
            for j in range(i):
                wj = top_idx[j]
                dij = int(term_doc[:, wi].multiply(term_doc[:, wj]).sum())
                dj = int(term_doc[:, wj].sum())
                scores.append(math.log((dij + 1) / max(1, dj)))
    return float(np.mean(scores)) if scores else 0.0

def heldout_macro_f1(texts, labels):
    counts = Counter(labels)
    if len(counts) < 2 or min(counts.values()) < 2:
        return np.nan
    test_size = max(len(counts), int(round(len(labels) * 0.3)))
    try:
        X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=test_size, stratify=labels, random_state=SEED)
        model = Pipeline([
            ("tfidf", TfidfVectorizer(min_df=1, ngram_range=(1, 2))),
            ("clf", LogisticRegression(max_iter=2000, random_state=SEED)),
        ])
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        return float(f1_score(y_test, pred, average="macro", zero_division=0))
    except Exception:
        return np.nan

phase3_texts = chunks_df.loc[~chunks_df["is_metadata"], "clean_text"].tolist()
phase3_labels = chunks_df.loc[~chunks_df["is_metadata"], "class_label"].tolist()
max_k = min(8, max(2, len(phase3_texts) - 1))
candidate_ks = list(range(2, max_k + 1)) if max_k >= 2 else [len(domain_classes)]
macro_f1_value = heldout_macro_f1(phase3_texts, phase3_labels)
phase3_rows = []
for k in candidate_ks:
    phase3_rows.append({
        "k": k,
        "lda_coherence": lda_umass_like_coherence(phase3_texts, k),
        "kb_macro_f1": macro_f1_value,
        "taxonomy_mode": taxonomy_mode,
    })
phase3_results = pd.DataFrame(phase3_rows)
if taxonomy_mode in ["folder", "top_heading"]:
    SELECTED_K = len(domain_classes)
else:
    score = phase3_results["lda_coherence"].rank(pct=True) + phase3_results["kb_macro_f1"].fillna(0).rank(pct=True)
    SELECTED_K = int(phase3_results.loc[score.idxmax(), "k"])
phase3_results.to_csv(OUTPUT_PATH / "phase3_k_selection.csv", index=False)
print(phase3_results.to_string(index=False))
print(f"Selected K: {SELECTED_K}")


 k  lda_coherence  kb_macro_f1 taxonomy_mode
 2      -0.350801          1.0        folder
 3      -0.123813          1.0        folder
 4      -0.021947          1.0        folder
 5      -0.041508          1.0        folder
 6       0.139158          1.0        folder
 7       0.139622          1.0        folder
Selected K: 2


## Phase 4. Classifier feature-representation benchmark

This phase benchmarks the representations specified in the v3 plan. It uses SLP3 Appendix B for Naive Bayes, SLP3 Chapters 4 and 5 for linear classifiers and TF-IDF, Mikolov et al. for Word2Vec, and Reimers and Gurevych for Sentence-BERT.

In [7]:
# Phase 4 implementation
class DenseMatrixEstimator(BaseEstimator):
    def __init__(self, vectors=None, classifier=None):
        self.vectors = vectors
        self.classifier = classifier if classifier is not None else LogisticRegression(max_iter=2000, random_state=SEED)
    def fit(self, X, y):
        indices = np.asarray(X, dtype=int)
        self.classifier_ = LogisticRegression(max_iter=2000, random_state=SEED)
        self.classifier_.fit(self.vectors[indices], y)
        return self
    def predict(self, X):
        indices = np.asarray(X, dtype=int)
        return self.classifier_.predict(self.vectors[indices])
    def predict_proba(self, X):
        indices = np.asarray(X, dtype=int)
        if hasattr(self.classifier_, "predict_proba"):
            return self.classifier_.predict_proba(self.vectors[indices])
        return np.zeros((len(indices), len(self.classifier_.classes_)))

def bootstrap_f1_ci(y_true, y_pred, average="macro", n_boot=500):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    if len(y_true) == 0:
        return (np.nan, np.nan)
    rng = np.random.default_rng(SEED)
    scores = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_true), len(y_true))
        scores.append(f1_score(y_true[idx], y_pred[idx], average=average, zero_division=0))
    return float(np.percentile(scores, 2.5)), float(np.percentile(scores, 97.5))

def make_word2vec_features(texts, token_lists):
    if gensim_models is not None:
        try:
            model = gensim_models.Word2Vec(sentences=token_lists, vector_size=50, window=5, min_count=1, workers=1, seed=SEED, epochs=100)
            vectors = []
            for tokens in token_lists:
                vecs = [model.wv[token] for token in tokens if token in model.wv]
                vectors.append(np.mean(vecs, axis=0) if vecs else np.zeros(model.vector_size))
            return np.asarray(vectors), "gensim_word2vec"
        except Exception as exc:
            print(f"[skip] reason: gensim Word2Vec failed ({exc.__class__.__name__}); using TF-IDF SVD dense fallback")
    vectors, _ = fit_dense_encoder(texts, model_name="word2vec-fallback")
    return vectors, "tfidf_svd_fallback_no_gensim"

def make_sbert_features(texts):
    vectors, encoder = fit_dense_encoder(texts, model_name="all-MiniLM-L6-v2")
    return vectors, encoder["kind"]

def make_openai_embedding_features(texts):
    if not OPENAI_API_KEY or openai_mod is None:
        return None, "skipped_no_key_or_package"
    try:
        client = openai_mod.OpenAI(api_key=OPENAI_API_KEY)
        response = client.embeddings.create(model="text-embedding-3-small", input=list(texts))
        vectors = np.asarray([item.embedding for item in response.data])
        return normalize(vectors), "openai_text_embedding_3_small"
    except Exception as exc:
        print(f"[skip] reason: OpenAI embeddings failed ({exc.__class__.__name__})")
        return None, "skipped_api_error"

def evaluate_estimator(name, X_input, y, estimator, cv, status):
    pred = cross_val_predict(estimator, X_input, y, cv=cv)
    macro = f1_score(y, pred, average="macro", zero_division=0)
    weighted = f1_score(y, pred, average="weighted", zero_division=0)
    ci_low, ci_high = bootstrap_f1_ci(y, pred, average="macro")
    per_class = {label: f1_score(np.asarray(y) == label, np.asarray(pred) == label, zero_division=0) for label in sorted(set(y))}
    return {
        "representation": name,
        "macro_f1": float(macro),
        "weighted_f1": float(weighted),
        "macro_f1_ci_low": ci_low,
        "macro_f1_ci_high": ci_high,
        "per_class_f1": json.dumps(per_class, sort_keys=True),
        "status": status,
    }

bench_df = chunks_df.loc[~chunks_df["is_metadata"]].reset_index(drop=True).copy()
X_text = bench_df["clean_text"].tolist()
y = bench_df["class_label"].tolist()
class_counts = Counter(y)
cv_splits = min(5, min(class_counts.values())) if class_counts else 0
if cv_splits < 2:
    raise ValueError("Phase 4 needs at least two KB chunks per class for smoke-test CV")
if cv_splits < 5:
    print(f"[skip] reason: synthetic class counts allow {cv_splits}-fold CV; real run uses 5-fold when every class has at least five chunks")
cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=SEED)
indices = np.arange(len(X_text))

phase4_rows = []
phase4_rows.append(evaluate_estimator(
    "4A Bag-of-Words + MultinomialNB",
    X_text,
    y,
    Pipeline([("count", CountVectorizer(min_df=1)), ("clf", MultinomialNB())]),
    cv,
    "ok",
))
phase4_rows.append(evaluate_estimator(
    "4B TF-IDF unigram + LogisticRegression",
    X_text,
    y,
    Pipeline([("tfidf", TfidfVectorizer(min_df=1, ngram_range=(1, 1))), ("clf", LogisticRegression(max_iter=2000, random_state=SEED))]),
    cv,
    "ok",
))
phase4_rows.append(evaluate_estimator(
    "4C TF-IDF unigram_bigram + LogisticRegression",
    X_text,
    y,
    Pipeline([("tfidf", TfidfVectorizer(min_df=1, ngram_range=(1, 2))), ("clf", LogisticRegression(max_iter=2000, random_state=SEED))]),
    cv,
    "ok",
))
phase4_rows.append(evaluate_estimator(
    "4D TF-IDF + LinearSVM",
    X_text,
    y,
    Pipeline([("tfidf", TfidfVectorizer(min_df=1, ngram_range=(1, 2))), ("clf", LinearSVC(random_state=SEED, max_iter=5000))]),
    cv,
    "ok",
))
w2v_vectors, w2v_status = make_word2vec_features(X_text, bench_df["tokens"].tolist())
phase4_rows.append(evaluate_estimator(
    "4E Word2Vec averaged + LogisticRegression",
    indices,
    y,
    DenseMatrixEstimator(vectors=w2v_vectors),
    cv,
    w2v_status,
))
sbert_vectors, sbert_status = make_sbert_features(X_text)
phase4_rows.append(evaluate_estimator(
    "4F Sentence-BERT all-MiniLM-L6-v2 + LogisticRegression",
    indices,
    y,
    DenseMatrixEstimator(vectors=sbert_vectors),
    cv,
    sbert_status,
))
openai_vectors, openai_status = make_openai_embedding_features(X_text)
if openai_vectors is not None:
    phase4_rows.append(evaluate_estimator(
        "4G OpenAI text-embedding-3-small + LogisticRegression",
        indices,
        y,
        DenseMatrixEstimator(vectors=openai_vectors),
        cv,
        openai_status,
    ))
else:
    print(f"[skip] reason: OpenAI row skipped ({openai_status})")

phase4_results = pd.DataFrame(phase4_rows).sort_values("macro_f1", ascending=False).reset_index(drop=True)
phase4_results.to_csv(OUTPUT_PATH / "phase4_results.csv", index=False)
winning_phase4 = phase4_results.iloc[0].to_dict()
print(phase4_results[["representation", "macro_f1", "weighted_f1", "macro_f1_ci_low", "macro_f1_ci_high", "status"]].to_string(index=False))
print(f"Winning representation: {winning_phase4['representation']}")

plt.rcParams.update({"font.family": "serif", "figure.facecolor": "white", "axes.facecolor": "white"})
fig, ax = plt.subplots(figsize=(6.4, 3.6))
plot_df = phase4_results.sort_values("macro_f1")
errors = [plot_df["macro_f1"] - plot_df["macro_f1_ci_low"], plot_df["macro_f1_ci_high"] - plot_df["macro_f1"]]
ax.barh(plot_df["representation"], plot_df["macro_f1"], xerr=errors, color="#4967AA", alpha=0.9)
ax.set_xlabel("Macro-F1")
ax.set_xlim(0, 1.0)
ax.grid(axis="x", color="#e9ecef")
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "fig04_classifier_benchmark.png", dpi=300, bbox_inches="tight", facecolor="white")
fig.savefig(FIGURE_DIR / "fig04_classifier_benchmark.pdf", dpi=300, bbox_inches="tight", facecolor="white")
plt.close(fig)


[skip] reason: synthetic class counts allow 4-fold CV; real run uses 5-fold when every class has at least five chunks
[skip] reason: OpenAI row skipped (skipped_no_key_or_package)
                                        representation  macro_f1  weighted_f1  macro_f1_ci_low  macro_f1_ci_high                       status
                       4A Bag-of-Words + MultinomialNB  0.873016     0.873016         0.466667               1.0                           ok
                4B TF-IDF unigram + LogisticRegression  0.873016     0.873016         0.466667               1.0                           ok
         4C TF-IDF unigram_bigram + LogisticRegression  0.873016     0.873016         0.466667               1.0                           ok
                                 4D TF-IDF + LinearSVM  0.873016     0.873016         0.466667               1.0                           ok
             4E Word2Vec averaged + LogisticRegression  0.619048     0.619048         0.250000               1

## Phase 5. Keywords per class

Top keywords come from TF-IDF coefficients, following SLP3 Chapter 5. If the winning representation has no interpretable coefficients, the notebook trains an interpretable TF-IDF logistic model for this export step only.

In [8]:
# Phase 5 implementation
def ensure_ticket_gold_labels(tickets, chunks):
    tickets = tickets.copy()
    missing = tickets["ticket_gold_label"].astype(str).str.len() == 0
    non_access_missing = missing & (~tickets["is_access_change_request"])
    if non_access_missing.any():
        vectorizer = TfidfVectorizer(min_df=1)
        corpus = chunks.loc[~chunks["is_metadata"], "clean_text"].tolist() + tickets.loc[non_access_missing, "clean_text"].tolist()
        matrix = vectorizer.fit_transform(corpus)
        chunk_matrix = matrix[:len(chunks.loc[~chunks["is_metadata"]])]
        ticket_matrix = matrix[len(chunks.loc[~chunks["is_metadata"]]):]
        chunk_labels = chunks.loc[~chunks["is_metadata"], "class_label"].tolist()
        centroids = []
        labels = []
        for label in sorted(set(chunk_labels)):
            idx = [i for i, value in enumerate(chunk_labels) if value == label]
            centroids.append(np.asarray(chunk_matrix[idx].mean(axis=0)).ravel())
            labels.append(label)
        sims = cosine_similarity(ticket_matrix, np.vstack(centroids))
        inferred = [labels[i] for i in sims.argmax(axis=1)]
        tickets.loc[non_access_missing, "ticket_gold_label"] = inferred
    tickets.loc[tickets["is_access_change_request"], "ticket_gold_label"] = "access_change_request"
    return tickets

tickets_df = ensure_ticket_gold_labels(tickets_df, chunks_df)
keyword_model = Pipeline([
    ("tfidf", TfidfVectorizer(min_df=1, ngram_range=(1, 2), stop_words="english")),
    ("clf", LogisticRegression(max_iter=2000, random_state=SEED)),
])
keyword_model.fit(X_text, y)
feature_names = np.asarray(keyword_model.named_steps["tfidf"].get_feature_names_out())
coef = keyword_model.named_steps["clf"].coef_
classes_for_keywords = keyword_model.named_steps["clf"].classes_
if coef.shape[0] == 1 and len(classes_for_keywords) == 2:
    coef = np.vstack([-coef[0], coef[0]])
keyword_rankings = {}
for idx, class_id in enumerate(classes_for_keywords):
    order = np.argsort(coef[idx])[::-1]
    keyword_rankings[class_id] = feature_names[order].tolist()

def keyword_predict(text, top_n):
    scores = {}
    text_tokens = set(simple_tokenize(text))
    text_string = f" {text} "
    for class_id, terms in keyword_rankings.items():
        score = 0
        for term in terms[:top_n]:
            if " " in term:
                score += int(f" {term} " in text_string)
            else:
                score += int(term in text_tokens)
        scores[class_id] = score
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "unmatched"

heldout = tickets_df[~tickets_df["is_access_change_request"]].copy()
precision_rows = []
for n in range(3, 31):
    preds = [keyword_predict(text, n) for text in heldout["clean_text"]]
    precision_rows.append({"n": n, "precision_macro": precision_score(heldout["ticket_gold_label"], preds, average="macro", zero_division=0)})
precision_df = pd.DataFrame(precision_rows)
if len(precision_df) >= 3:
    gains = precision_df["precision_macro"].diff().fillna(precision_df["precision_macro"])
    elbow_candidates = precision_df.loc[gains <= 0.01, "n"]
    KEYWORD_N = int(elbow_candidates.iloc[0]) if len(elbow_candidates) else int(precision_df.loc[precision_df["precision_macro"].idxmax(), "n"])
else:
    KEYWORD_N = 10

keyword_rows = []
for class_id, terms in keyword_rankings.items():
    keyword_rows.append({"class_id": class_id, "n": KEYWORD_N, "keywords": json.dumps(terms[:KEYWORD_N])})
keywords_df = pd.DataFrame(keyword_rows)
keywords_df.to_csv(OUTPUT_PATH / "phase5_keywords.csv", index=False)
precision_df.to_csv(OUTPUT_PATH / "phase5_keyword_precision_curve.csv", index=False)
print(f"Selected keyword N: {KEYWORD_N}")
print(keywords_df.to_string(index=False))


Selected keyword N: 4
       class_id  n                                          keywords
access_security  4            ["access", "role", "reset", "browser"]
  close_process  4 ["translation", "adjustment", "partner", "close"]


## Phase 6. Architecture comparison

This phase implements the v3 retrieval comparison with chunk-level dense retrieval. It covers classifier routing, pure RAG, classifier plus reranking, and a classifier plus RAG hybrid. The design cites SLP3 Chapter 9, Lewis et al. for RAG, Reimers and Gurevych for sentence embeddings, and Pradeep et al. for reranking.

In [9]:
# Phase 6 implementation
def bootstrap_rate_ci(hits, n_boot=1000):
    hits = np.asarray(hits, dtype=float)
    if len(hits) == 0:
        return (np.nan, np.nan)
    rng = np.random.default_rng(SEED)
    rates = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(hits), len(hits))
        rates.append(float(hits[idx].mean()))
    return float(np.percentile(rates, 2.5)), float(np.percentile(rates, 97.5))

routing_vectorizer = TfidfVectorizer(min_df=1, ngram_range=(1, 2), stop_words="english")
routing_X = routing_vectorizer.fit_transform(bench_df["clean_text"])
routing_clf = LogisticRegression(max_iter=2000, random_state=SEED)
routing_clf.fit(routing_X, y)
routing_classes = list(routing_clf.classes_)

chunk_vectors, dense_encoder = fit_dense_encoder(chunks_df.loc[~chunks_df["is_metadata"], "clean_text"].tolist(), model_name="all-MiniLM-L6-v2")
retrieval_chunks = chunks_df.loc[~chunks_df["is_metadata"]].reset_index(drop=True).copy()
ticket_vectors = transform_dense_encoder(tickets_df["clean_text"].tolist(), dense_encoder)
chunk_sim = cosine_similarity(ticket_vectors, chunk_vectors)

if USE_FAISS and faiss_mod is not None:
    try:
        faiss_index = faiss_mod.IndexFlatIP(chunk_vectors.shape[1])
        faiss_index.add(chunk_vectors.astype("float32"))
        print("FAISS IndexFlatIP enabled for dense retrieval")
    except Exception as exc:
        faiss_index = None
        print(f"[skip] reason: FAISS setup failed ({exc.__class__.__name__}); using sklearn cosine")
else:
    faiss_index = None
    if USE_FAISS:
        print("[skip] reason: FAISS requested but unavailable; using sklearn cosine")

class_centroids = {}
for class_id in domain_classes:
    idx = retrieval_chunks.index[retrieval_chunks["class_label"] == class_id].tolist()
    class_centroids[class_id] = normalize(chunk_vectors[idx].mean(axis=0).reshape(1, -1))[0]
centroid_matrix = np.vstack([class_centroids[class_id] for class_id in domain_classes])
chunk_to_centroid_sim = cosine_similarity(chunk_vectors, centroid_matrix)
membership_matrix = chunk_to_centroid_sim >= MEMBERSHIP_THRESHOLD
for row_idx, class_id in enumerate(retrieval_chunks["class_label"]):
    if class_id in domain_classes:
        membership_matrix[row_idx, domain_classes.index(class_id)] = True
docs_by_class = {}
for class_idx, class_id in enumerate(domain_classes):
    docs_by_class[class_id] = np.where(membership_matrix[:, class_idx])[0].tolist()

reranker_status = "lexical_tfidf_fallback"
cross_encoder_model = None
if CrossEncoder is not None:
    try:
        cross_encoder_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
        reranker_status = "cross-encoder/ms-marco-MiniLM-L-6-v2"
    except Exception as exc:
        print(f"[skip] reason: cross-encoder unavailable ({exc.__class__.__name__}); using lexical reranker")
else:
    print("[skip] reason: cross-encoder package unavailable; using lexical reranker")

def classifier_probs(query_clean):
    Xq = routing_vectorizer.transform([query_clean])
    probs = routing_clf.predict_proba(Xq)[0]
    return dict(zip(routing_classes, probs))

def dense_rank(query_idx, candidate_indices=None, limit=10):
    if candidate_indices is None:
        candidate_indices = list(range(len(retrieval_chunks)))
    candidate_indices = list(dict.fromkeys([int(i) for i in candidate_indices if 0 <= int(i) < len(retrieval_chunks)]))
    if not candidate_indices:
        candidate_indices = list(range(len(retrieval_chunks)))
    scores = chunk_sim[query_idx, candidate_indices]
    order = np.argsort(scores)[::-1]
    return [candidate_indices[i] for i in order[:limit]]

def rerank_candidates(query_text, candidate_indices, limit=10):
    candidate_indices = list(candidate_indices)
    if not candidate_indices:
        return []
    if cross_encoder_model is not None:
        pairs = [(query_text, retrieval_chunks.loc[idx, "chunk_text"]) for idx in candidate_indices]
        scores = cross_encoder_model.predict(pairs)
        order = np.argsort(scores)[::-1]
        return [candidate_indices[i] for i in order[:limit]]
    vectorizer = TfidfVectorizer(min_df=1)
    texts = [query_text] + [retrieval_chunks.loc[idx, "clean_text"] for idx in candidate_indices]
    mat = vectorizer.fit_transform(texts)
    scores = cosine_similarity(mat[0], mat[1:]).ravel()
    order = np.argsort(scores)[::-1]
    return [candidate_indices[i] for i in order[:limit]]

def architecture_candidates(row, query_idx, arch, limit=10):
    if row["is_access_change_request"]:
        return []
    probs = classifier_probs(row["clean_text"])
    sorted_classes = sorted(probs, key=probs.get, reverse=True)
    top_class = sorted_classes[0]
    if arch == "A flat classifier to class KB":
        return dense_rank(query_idx, docs_by_class.get(top_class, []), limit=limit)
    if arch == "B two-stage hierarchical":
        stage1 = "navigation" if re.search(r"\b(url|link|login|access|portal)\b", row["clean_text"]) else "informational"
        if stage1 == "navigation" and "access_security" in docs_by_class:
            return dense_rank(query_idx, docs_by_class["access_security"], limit=limit)
        return dense_rank(query_idx, docs_by_class.get(top_class, []), limit=limit)
    if arch == "C multi-label fan-out cosine merge":
        chosen = [class_id for class_id, prob in probs.items() if prob >= MULTILABEL_THRESHOLD]
        if not chosen:
            chosen = sorted_classes[:2]
        pool = []
        for class_id in chosen:
            pool.extend(docs_by_class.get(class_id, []))
        return dense_rank(query_idx, pool, limit=limit)
    if arch == "D pure RAG chunk dense retrieval":
        return dense_rank(query_idx, None, limit=limit)
    if arch == "E classifier plus cross-encoder reranker":
        initial = dense_rank(query_idx, docs_by_class.get(top_class, []), limit=max(10, limit))
        return rerank_candidates(row["clean_text"], initial, limit=limit)
    if arch == "F classifier plus RAG hybrid":
        chosen = sorted_classes[:2]
        pool = []
        for class_id in chosen:
            pool.extend(docs_by_class.get(class_id, []))
        return dense_rank(query_idx, pool, limit=limit)
    return []

def optional_llm_answer(query, chunk_records):
    if not RUN_OPTIONAL_LLM_SAMPLE or openai_mod is None or not OPENAI_API_KEY:
        return "[skip] reason: OPENAI_API_KEY or openai package unavailable"
    context = "\n\n".join([record["chunk_text_redacted"] for record in chunk_records[:3]])
    try:
        client = openai_mod.OpenAI(api_key=OPENAI_API_KEY)
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "Answer using only the supplied OneStream KB excerpts."},
                {"role": "user", "content": f"Query: {query}\n\nKB excerpts:\n{context}"},
            ],
            temperature=0,
        )
        return response.choices[0].message.content
    except Exception as exc:
        return f"[skip] reason: LLM call failed ({exc.__class__.__name__})"

architectures = [
    "A flat classifier to class KB",
    "B two-stage hierarchical",
    "C multi-label fan-out cosine merge",
    "D pure RAG chunk dense retrieval",
    "E classifier plus cross-encoder reranker",
    "F classifier plus RAG hybrid",
]
ks = [1, 3, 10]
eval_tickets = tickets_df[tickets_df["ticket_gold_label"].isin(domain_classes)].reset_index(drop=True)
phase6_rows = []
for arch in architectures:
    all_rankings = []
    for local_idx, row in eval_tickets.iterrows():
        original_idx = tickets_df.index[tickets_df["ticket_id"] == row["ticket_id"]][0]
        ranking = architecture_candidates(row, original_idx, arch, limit=max(ks))
        all_rankings.append(ranking)
    for k in ks:
        hits = []
        for (_, row), ranking in zip(eval_tickets.iterrows(), all_rankings):
            returned_labels = retrieval_chunks.loc[ranking[:k], "class_label"].tolist() if ranking else []
            hits.append(row["ticket_gold_label"] in returned_labels)
        ci_low, ci_high = bootstrap_rate_ci(hits)
        phase6_rows.append({
            "architecture": arch,
            "k": k,
            "n_eval": len(hits),
            "retrieval_at_k": float(np.mean(hits)) if hits else np.nan,
            "ci_low": ci_low,
            "ci_high": ci_high,
            "dense_encoder": dense_encoder["kind"],
            "reranker": reranker_status if "reranker" in arch else "not_used",
        })
phase6_results = pd.DataFrame(phase6_rows)
phase6_results.to_csv(OUTPUT_PATH / "phase6_results.csv", index=False)
print(phase6_results.to_string(index=False))

if RUN_OPTIONAL_LLM_SAMPLE and len(eval_tickets):
    llm_rows = []
    for _, row in eval_tickets.head(LLM_SAMPLE_SIZE).iterrows():
        original_idx = tickets_df.index[tickets_df["ticket_id"] == row["ticket_id"]][0]
        ranking = architecture_candidates(row, original_idx, "D pure RAG chunk dense retrieval", limit=3)
        chunks = retrieval_chunks.loc[ranking].to_dict("records")
        llm_rows.append({"ticket_id": row["ticket_id"], "answer": optional_llm_answer(row["ticket_text_redacted"], chunks)})
    pd.DataFrame(llm_rows).to_csv(OUTPUT_PATH / "phase6_optional_llm_sample.csv", index=False)

plot6 = phase6_results[phase6_results["k"] == 3].copy().sort_values("retrieval_at_k")
fig, ax = plt.subplots(figsize=(6.4, 3.8))
errors = [plot6["retrieval_at_k"] - plot6["ci_low"], plot6["ci_high"] - plot6["retrieval_at_k"]]
ax.barh(plot6["architecture"], plot6["retrieval_at_k"], xerr=errors, color="#8E5A7A", alpha=0.92)
ax.set_xlabel("Retrieval@3")
ax.set_xlim(0, 1.0)
ax.grid(axis="x", color="#e9ecef")
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "fig06_architecture_comparison.png", dpi=300, bbox_inches="tight", facecolor="white")
fig.savefig(FIGURE_DIR / "fig06_architecture_comparison.pdf", dpi=300, bbox_inches="tight", facecolor="white")
plt.close(fig)


[skip] reason: cross-encoder package unavailable; using lexical reranker
                            architecture  k  n_eval  retrieval_at_k  ci_low  ci_high dense_encoder               reranker
           A flat classifier to class KB  1      30             1.0     1.0      1.0     tfidf_svd               not_used
           A flat classifier to class KB  3      30             1.0     1.0      1.0     tfidf_svd               not_used
           A flat classifier to class KB 10      30             1.0     1.0      1.0     tfidf_svd               not_used
                B two-stage hierarchical  1      30             1.0     1.0      1.0     tfidf_svd               not_used
                B two-stage hierarchical  3      30             1.0     1.0      1.0     tfidf_svd               not_used
                B two-stage hierarchical 10      30             1.0     1.0      1.0     tfidf_svd               not_used
      C multi-label fan-out cosine merge  1      30             1.0     1

## Phase 7. Knowledge-base consolidation

This phase checks class-pair Jaccard overlap on the multi-membership matrix and flags orphan documents, following the v3 consolidation plan.

In [10]:
# Phase 7 implementation
jaccard_rows = []
for i, class_a in enumerate(domain_classes):
    set_a = set(np.where(membership_matrix[:, i])[0])
    for j, class_b in enumerate(domain_classes):
        if j <= i:
            continue
        set_b = set(np.where(membership_matrix[:, j])[0])
        union = set_a | set_b
        jaccard = len(set_a & set_b) / len(union) if union else 0.0
        jaccard_rows.append({"class_a": class_a, "class_b": class_b, "jaccard": jaccard})
jaccard_df = pd.DataFrame(jaccard_rows)
claimed = membership_matrix.any(axis=1)
orphan_chunks = retrieval_chunks.loc[~claimed, ["chunk_id", "relative_path", "filename", "chunk_text_redacted"]].copy()
orphan_docs = sorted(orphan_chunks["relative_path"].unique().tolist()) if len(orphan_chunks) else []
jaccard_df.to_csv(OUTPUT_PATH / "phase7_class_pair_jaccard.csv", index=False)
orphan_chunks.to_csv(OUTPUT_PATH / "phase7_orphan_chunks.csv", index=False)
print("Class-pair Jaccard")
print(jaccard_df.to_string(index=False) if len(jaccard_df) else "No class pairs")
print(f"Orphan documents: {orphan_docs}")


Class-pair Jaccard
        class_a       class_b  jaccard
access_security close_process    0.125
Orphan documents: []


## Phase 8. Dify export

The final phase writes Dify configuration and runtime metadata. The export contains evaluation metrics, class entries, the access redirect class, and orphan-document flags. Raw ticket text is not written.

In [11]:
# Phase 8 implementation
phase6_r3 = phase6_results[phase6_results["k"] == 3].sort_values("retrieval_at_k", ascending=False).head(1)
best_r3 = phase6_r3.iloc[0].to_dict() if len(phase6_r3) else {}
phase4_best = phase4_results.sort_values("macro_f1", ascending=False).iloc[0].to_dict()
keywords_map = {row["class_id"]: json.loads(row["keywords"]) for _, row in keywords_df.iterrows()}
class_entries = []
for class_id in domain_classes:
    member_idx = docs_by_class.get(class_id, [])
    member_docs = sorted(retrieval_chunks.loc[member_idx, "relative_path"].unique().tolist()) if member_idx else []
    class_entries.append({
        "class_id": class_id,
        "keywords": keywords_map.get(class_id, []),
        "kb_documents": member_docs,
        "redirect_url": "",
    })
class_entries.append({
    "class_id": "access_change_request",
    "keywords": ["change request", "role access", "request access", "access portal"],
    "kb_documents": [],
    "redirect_url": ACCESS_REDIRECT_URL,
})

dify_config = {
    "metadata": {
        "created_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
        "seed": SEED,
        "selected_k": SELECTED_K,
        "keyword_n": KEYWORD_N,
        "taxonomy_mode": taxonomy_mode,
        "embedding_model": "all-MiniLM-L6-v2" if dense_encoder["kind"] == "sentence_transformer" else "tfidf_svd_fallback",
        "classifier": phase4_best["representation"],
        "recommended_architecture": best_r3.get("architecture", ""),
    },
    "classes": class_entries,
    "orphan_documents": orphan_docs,
    "evaluation": {
        "coherence": float(phase3_results["lda_coherence"].max()) if len(phase3_results) else np.nan,
        "macro_f1": float(phase4_best["macro_f1"]),
        "macro_f1_ci": [float(phase4_best["macro_f1_ci_low"]), float(phase4_best["macro_f1_ci_high"])],
        "retrieval_at_3": float(best_r3.get("retrieval_at_k", np.nan)),
        "retrieval_at_3_ci": [float(best_r3.get("ci_low", np.nan)), float(best_r3.get("ci_high", np.nan))],
    },
}
with (OUTPUT_PATH / "dify_config.json").open("w", encoding="utf-8") as f:
    json.dump(dify_config, f, indent=2)

metadata_docs = kb_docs_df[kb_docs_df["is_metadata"]].copy()
runtime_metadata = {
    "created_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "metadata_documents": [
        {"relative_path": row["relative_path"], "top_heading": row["top_heading"], "doc_id": row["doc_id"]}
        for _, row in metadata_docs.iterrows()
    ],
}
with (OUTPUT_PATH / "runtime_metadata.json").open("w", encoding="utf-8") as f:
    json.dump(runtime_metadata, f, indent=2)
print(f"Wrote {OUTPUT_PATH / 'dify_config.json'}")
print(f"Wrote {OUTPUT_PATH / 'runtime_metadata.json'}")


Wrote dify_exports/dify_config.json
Wrote dify_exports/runtime_metadata.json
